# FOTW Autoresearch Analysis

Interactive exploration of skill optimization experiments.

Set `TARGET` below and run all cells.

In [ ]:
# === CONFIGURATION ===
TARGET = "desloppify"  # Change this to analyze a different target
# =====================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

EVALS_DIR = Path(".") if Path("targets").exists() else Path("evals")
TSV = EVALS_DIR / "targets" / TARGET / "results.tsv"

df = pd.read_csv(TSV, sep="\t")
df["pass_rate"] = df["pass_rate"].str.rstrip("%").astype(float)
df["status"] = df["status"].str.strip().str.lower()

print(f"Target: {TARGET}")
print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
# Experiment outcomes
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("keep", 0)
n_discard = counts.get("discard", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# All KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "keep"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    rate = row["pass_rate"]
    desc = row["description"]
    print(f"  #{i:3d}  rate={rate:.1f}%  score={row['score']}/{row['max_score']}  {desc}")

In [ ]:
# Pass Rate Over Time
fig, ax = plt.subplots(figsize=(16, 8))

baseline_rate = df.iloc[0]["pass_rate"]

# Discarded: gray
disc = df[df["status"] == "discard"]
if not disc.empty:
    ax.scatter(disc.index, disc["pass_rate"],
               c="#cccccc", s=15, alpha=0.5, zorder=2, label="Discarded")

# Kept: green
if not kept.empty:
    ax.scatter(kept.index, kept["pass_rate"],
               c="#2ecc71", s=60, zorder=4, label="Kept",
               edgecolors="black", linewidths=0.5)

    # Running maximum
    running_max = kept["pass_rate"].cummax()
    ax.step(kept.index, running_max, where="post", color="#27ae60",
            linewidth=2, alpha=0.7, zorder=3, label="Running best")

    # Labels
    for idx, row in kept.iterrows():
        desc = str(row["description"]).strip()
        if len(desc) > 45:
            desc = desc[:42] + "..."
        ax.annotate(desc, (idx, row["pass_rate"]),
                    textcoords="offset points", xytext=(6, 6),
                    fontsize=8, color="#1a7a3a", alpha=0.9,
                    rotation=25, ha="left", va="bottom")

# Baseline
ax.axhline(y=baseline_rate, color="#3498db", linewidth=1, linestyle="--",
           alpha=0.5, label=f"Baseline ({baseline_rate:.1f}%)")

best_rate = kept["pass_rate"].max() if not kept.empty else baseline_rate
n_total = len(df)
n_kept = len(kept)
improvement = best_rate - baseline_rate

ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Pass Rate % (higher is better)", fontsize=12)
ax.set_title(
    f"{TARGET}: {n_total} Experiments, {n_kept} Kept, "
    f"Best {best_rate:.1f}% ({improvement:+.1f}%)",
    fontsize=14,
)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)
ax.set_ylim(max(0, baseline_rate - 15), min(100, best_rate + 15))

plt.tight_layout()
plt.show()

In [ ]:
# Summary Statistics
baseline_rate = df.iloc[0]["pass_rate"]
best_rate = kept["pass_rate"].max() if not kept.empty else baseline_rate
best_row = kept.loc[kept["pass_rate"].idxmax()] if not kept.empty else df.iloc[0]

print(f"Baseline pass rate:  {baseline_rate:.1f}%")
print(f"Best pass rate:      {best_rate:.1f}%")
print(f"Total improvement:   {best_rate - baseline_rate:+.1f}%")
print(f"Best experiment:     {best_row['description']}")
print()

# Effort per improvement
print("Cumulative effort per improvement:")
for i, (_, row) in enumerate(kept.iterrows()):
    print(f"  Experiment #{_:3d}: rate={row['pass_rate']:.1f}%  {row['description']}")

In [ ]:
# Top Hits — Kept experiments ranked by improvement delta
if len(kept) > 1:
    hits = kept.copy()
    hits["prev_rate"] = hits["pass_rate"].shift(1)
    hits["delta"] = hits["pass_rate"] - hits["prev_rate"]
    hits = hits.iloc[1:]  # Drop baseline
    hits = hits.sort_values("delta", ascending=False)

    print(f"{'Rank':>4}  {'Delta':>8}  {'Rate':>8}  Description")
    print("-" * 75)
    for rank, (_, row) in enumerate(hits.iterrows(), 1):
        print(f"{rank:4d}  {row['delta']:+.1f}%  {row['pass_rate']:.1f}%  {row['description']}")
    print(f"\n{'':>4}  {hits['delta'].sum():+.1f}%  {'':>8}  TOTAL improvement over baseline")
else:
    print("Not enough kept experiments for delta analysis yet.")

In [ ]:
# Score Efficiency — score per experiment over time
if len(df) > 1:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Left: score/max_score ratio over time
    ax1.bar(df.index, df["score"], color=["#2ecc71" if s == "keep" else "#cccccc" for s in df["status"]])
    ax1.set_xlabel("Experiment #")
    ax1.set_ylabel("Raw Score")
    ax1.set_title("Raw Score Per Experiment")
    ax1.grid(True, alpha=0.15, axis="y")

    # Right: cumulative kept vs discarded
    cumulative = pd.DataFrame({
        "kept": (df["status"] == "keep").cumsum(),
        "discarded": (df["status"] == "discard").cumsum(),
    })
    ax2.fill_between(cumulative.index, cumulative["kept"], color="#2ecc71", alpha=0.3, label="Kept")
    ax2.fill_between(cumulative.index, cumulative["kept"], cumulative["kept"] + cumulative["discarded"],
                     color="#e74c3c", alpha=0.2, label="Discarded")
    ax2.plot(cumulative.index, cumulative["kept"], color="#27ae60", linewidth=2)
    ax2.set_xlabel("Experiment #")
    ax2.set_ylabel("Cumulative Count")
    ax2.set_title("Keep vs Discard Over Time")
    ax2.legend()
    ax2.grid(True, alpha=0.15)

    plt.tight_layout()
    plt.show()
else:
    print("Need more experiments for efficiency charts.")